# 01 Symbol Backtest

Notebook n?y d?ng ?? ki?m tra **m?t symbol** c?a chi?n l??c Combo t? ??u t?i cu?i:
- load data t? local DB
- ch?y backtest ??y ??
- xem metrics, equity, trade log
- ki?m tra Monte Carlo robustness


In [ ]:
# Bootstrap: add repo root + core_python to sys.path
import sys
from pathlib import Path

def _find_root(start: Path, marker: str = 'config.py') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f'Could not locate repo root containing {marker!r}')

ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('dark_background')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

from shared.monte_carlo import plot_monte_carlo, run_monte_carlo
from shared.theme import NUM_FMT
from strategies.combo.config import SYMBOLS, TIMEFRAME, get_indicator_params, summary as strategy_summary
from strategies.combo.symbol.backtest import run_symbol_backtest

print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
SYMBOL = 'US30'
ACCOUNT_MODE = 'standard'   # 'standard' | 'ftmo'
INITIAL_BALANCE = 100_000.0
DATE_FROM = '2023-01-01'
DATE_TO = None
MAX_BARS = 30000

INDICATOR_OVERRIDES = {
    # 'MA_PERIOD': 20,
    # 'KTP': 2.3,
    # 'MIN_RR': 1.25,
}

SYMBOL_OVERRIDES = {
    # 'x': 10.0,
    # 'ktp': 2.3,
    # 'ma_period': 20,
    # 'trailing_activation': 1.0,
}


In [ ]:
result = run_symbol_backtest(
    SYMBOL,
    init_eq=INITIAL_BALANCE,
    account_mode=ACCOUNT_MODE,
    date_from=DATE_FROM,
    date_to=DATE_TO,
    max_bars=MAX_BARS,
    indicator_overrides=INDICATOR_OVERRIDES or None,
    symbol_overrides=SYMBOL_OVERRIDES or None,
)

print('Done:', result.symbol, '| mode =', result.account_mode)
print('Trades =', len(result.trades))
print('Signal rows =', len(result.signal_data))


In [ ]:
metrics = pd.Series({k: v for k, v in result.metrics.items() if k != 'monthly_pnl_table'})
display(metrics.to_frame('value'))

monthly = result.metrics.get('monthly_pnl_table')
if isinstance(monthly, pd.DataFrame) and not monthly.empty:
    display(monthly)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

result.equity.plot(ax=axes[0], color='#00D4FF', lw=1.8, title=f'{SYMBOL} equity ({ACCOUNT_MODE})')
axes[0].grid(alpha=0.3)

if result.trades:
    trades_df = pd.DataFrame(result.trades)
    pnl_series = trades_df['pnl_usd'].cumsum()
    pnl_series.plot(ax=axes[1], color='#6BCB77', lw=1.5, title='Cumulative trade PnL')
    axes[1].grid(alpha=0.3)
else:
    axes[1].set_title('No trades')

plt.tight_layout()
plt.show()


In [ ]:
trade_cols = [
    'entry_time', 'exit_time', 'side', 'entry', 'sl', 'tp',
    'pnl_usd', 'r_multiple', 'bars_held', 'exit_reason'
]
if result.trades:
    trades_df = pd.DataFrame(result.trades)
    cols = [c for c in trade_cols if c in trades_df.columns]
    display(trades_df[cols].tail(30))
else:
    print('No trades to display.')


In [ ]:
if result.trades:
    trade_pnls = [float(t['pnl_usd']) for t in result.trades]
    mc = run_monte_carlo(trade_pnls, n_iter=500, dd_threshold=0.20)
    print('Monte Carlo P(max DD > 20%) =', round(mc['prob_exceed_dd'] * 100, 2), '%')
    print('Sharpe CI 95% =', (round(mc['sharpe_ci_low'], 2), round(mc['sharpe_ci_high'], 2)))
    plot_monte_carlo(mc)
else:
    print('Skip Monte Carlo because there are no trades.')
